Allows review of the data in the imageReviewer database 

This was generated using Claude Sonnet 4.5 on Typing Mind.

It is not yet validated for function, but it does create a working Gradio interface.

In [2]:
import gradio as gr
import sqlite3
import json
import os
from pathlib import Path
from datetime import datetime
import hashlib
from PIL import Image
import pandas as pd

In [3]:
class ImageRatingDB:
    def __init__(self, db_path="image_ratings.db"):
        self.db_path = db_path
        self.init_db()
    
    def init_db(self):
        """Initialize the database with required tables"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS image_ratings (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                unique_id TEXT UNIQUE NOT NULL,
                image_path TEXT NOT NULL,
                
                -- Rating criteria (0=N/A, 1-5 scale)
                aesthetic_quality INTEGER DEFAULT 0,
                prompt_adherence INTEGER DEFAULT 0,
                technical_quality INTEGER DEFAULT 0,
                style_consistency INTEGER DEFAULT 0,
                
                -- Prompts
                positive_prompt TEXT,
                negative_prompt TEXT,
                
                -- Generation parameters
                checkpoint_model TEXT,
                loras TEXT,
                cfg_scale REAL,
                seed INTEGER,
                height INTEGER,
                width INTEGER,
                clip_skip INTEGER,
                sampler TEXT,
                scheduler TEXT,
                
                -- Metadata
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        ''')
        
        conn.commit()
        conn.close()


    def generate_unique_id(self, json_data):
        """Generate a unique identifier from the generation parameters"""
        # Create a hash from key generation parameters
        key_params = {
            'seed': json_data.get('seed', ''),
            'prompt': json_data.get('prompt', ''),
            'negative_prompt': json_data.get('negative_prompt', ''),
            'cfg_scale': json_data.get('cfg_scale', ''),
            'steps': json_data.get('steps', ''),
            'sampler': json_data.get('sampler_name', ''),
            'model': json_data.get('sd_model_name', '')
        }
        
        hash_string = json.dumps(key_params, sort_keys=True)
        return hashlib.sha256(hash_string.encode()).hexdigest()

    def get_rating(self, unique_id):
        """Retrieve existing rating for an image"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute('''
            SELECT aesthetic_quality, prompt_adherence, technical_quality, 
                   style_consistency
            FROM image_ratings 
            WHERE unique_id = ?
        ''', (unique_id,))
        
        result = cursor.fetchone()
        conn.close()
        
        if result:
            return {
                'aesthetic_quality': result[0],
                'prompt_adherence': result[1],
                'technical_quality': result[2],
                'style_consistency': result[3]
            }
        return None


    def save_rating(self, unique_id, image_path, ratings, json_data):
        """Save or update a rating"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # Extract LORA information if present
        loras_str = json.dumps(json_data.get('loras', []))
        
        cursor.execute('''
            INSERT INTO image_ratings (
                unique_id, image_path,
                aesthetic_quality, prompt_adherence, technical_quality, style_consistency,
                positive_prompt, negative_prompt,
                checkpoint_model, loras, cfg_scale, seed, height, width,
                clip_skip, sampler, scheduler
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(unique_id) DO UPDATE SET
                image_path = excluded.image_path,
                aesthetic_quality = excluded.aesthetic_quality,
                prompt_adherence = excluded.prompt_adherence,
                technical_quality = excluded.technical_quality,
                style_consistency = excluded.style_consistency,
                updated_at = CURRENT_TIMESTAMP
        ''', (
            unique_id, image_path,
            ratings['aesthetic_quality'], ratings['prompt_adherence'],
            ratings['technical_quality'], ratings['style_consistency'],
            json_data.get('prompt', ''),
            json_data.get('negative_prompt', ''),
            json_data.get('sd_model_name', ''),
            loras_str,
            json_data.get('cfg_scale', 0),
            json_data.get('seed', 0),
            json_data.get('height', 0),
            json_data.get('width', 0),
            json_data.get('clip_skip', 0),
            json_data.get('sampler_name', ''),
            json_data.get('scheduler', '')
        ))
        
        conn.commit()
        conn.close()


    def get_all_ratings(self):
        """Get all ratings for analysis"""
        conn = sqlite3.connect(self.db_path)
        df = pd.read_sql_query("SELECT * FROM image_ratings", conn)
        conn.close()
        return df

    

In [4]:
#Initialize database
db = ImageRatingDB()

# Global variables to store current session data
current_image_path = None
current_info = None
current_unique_id = None

In [5]:
def get_summary_stats():
    """Generate summary statistics"""
    df = db.get_all_ratings()
    
    if df.empty:
        return "No ratings available yet."
    
    # Filter out N/A ratings (0) for meaningful statistics
    rating_cols = ['aesthetic_quality', 'prompt_adherence', 'technical_quality', 'style_consistency']
    
    stats = []
    stats.append("# 📈 Overall Statistics\n")
    stats.append(f"**Total Rated Images:** {len(df)}\n")
    
    for col in rating_cols:
        col_name = col.replace('_', ' ').title()
        valid_ratings = df[df[col] > 0][col]
        if len(valid_ratings) > 0:
            stats.append(f"\n**{col_name}:**")
            stats.append(f"- Average: {valid_ratings.mean():.2f}")
            stats.append(f"- Median: {valid_ratings.median():.0f}")
            stats.append(f"- Rated: {len(valid_ratings)}/{len(df)}")
    
    return "\n".join(stats)


In [6]:
def get_model_analysis():
    """Analyze performance by model"""
    df = db.get_all_ratings()
    
    if df.empty:
        return pd.DataFrame()
    
    # Group by model
    rating_cols = ['aesthetic_quality', 'prompt_adherence', 'technical_quality', 'style_consistency']
    
    model_stats = df.groupby('checkpoint_model')[rating_cols].agg(['mean', 'count'])
    model_stats.columns = ['_'.join(col).strip() for col in model_stats.columns.values]
    
    # Calculate overall average
    for col in rating_cols:
        valid_mask = df[col] > 0
        if valid_mask.any():
            model_stats[f'{col}_avg'] = df[valid_mask].groupby('checkpoint_model')[col].mean()
    
    model_stats = model_stats.reset_index()
    model_stats = model_stats.sort_values('aesthetic_quality_mean', ascending=False)
    
    return model_stats


In [7]:
def get_lora_analysis():
    """Analyze performance by LORA"""
    df = db.get_all_ratings()
    
    if df.empty:
        return pd.DataFrame()
    
    # Parse LORA JSON strings
    lora_ratings = []
    
    for idx, row in df.iterrows():
        try:
            loras = json.loads(row['loras']) if row['loras'] else []
            for lora in loras:
                lora_name = lora.get('name', 'Unknown')
                lora_ratings.append({
                    'lora': lora_name,
                    'aesthetic_quality': row['aesthetic_quality'],
                    'prompt_adherence': row['prompt_adherence'],
                    'technical_quality': row['technical_quality'],
                    'style_consistency': row['style_consistency']
                })
        except:
            continue
    
    if not lora_ratings:
        return pd.DataFrame({'message': ['No LORA data available']})
    
    lora_df = pd.DataFrame(lora_ratings)
    rating_cols = ['aesthetic_quality', 'prompt_adherence', 'technical_quality', 'style_consistency']
    
    lora_stats = lora_df.groupby('lora')[rating_cols].agg(['mean', 'count'])
    lora_stats.columns = ['_'.join(col).strip() for col in lora_stats.columns.values]
    lora_stats = lora_stats.reset_index()
    lora_stats = lora_stats.sort_values('aesthetic_quality_mean', ascending=False)
    
    return lora_stats

In [8]:
def get_best_worst_images():
    """Get best and worst rated images"""
    df = db.get_all_ratings()
    
    if df.empty:
        return "No ratings available", "No ratings available"
    
    # Calculate average rating (excluding N/A)
    rating_cols = ['aesthetic_quality', 'prompt_adherence', 'technical_quality', 'style_consistency']
    
    df['avg_rating'] = df[rating_cols].apply(
        lambda row: row[row > 0].mean() if (row > 0).any() else 0,
        axis=1
    )
    
    df_rated = df[df['avg_rating'] > 0].copy()
    
    if df_rated.empty:
        return "No fully rated images", "No fully rated images"
    
    # Best images
    best = df_rated.nlargest(5, 'avg_rating')[['image_path', 'avg_rating', 'checkpoint_model']]
    best_text = "# 🏆 Top 5 Images\n\n"
    for idx, row in best.iterrows():
        best_text += f"**{os.path.basename(row['image_path'])}**\n"
        best_text += f"- Average Rating: {row['avg_rating']:.2f}\n"
        best_text += f"- Model: {row['checkpoint_model']}\n\n"
    
    # Worst images
    worst = df_rated.nsmallest(5, 'avg_rating')[['image_path', 'avg_rating', 'checkpoint_model']]
    worst_text = "# 📉 Bottom 5 Images\n\n"
    for idx, row in worst.iterrows():
        worst_text += f"**{os.path.basename(row['image_path'])}**\n"
        worst_text += f"- Average Rating: {row['avg_rating']:.2f}\n"
        worst_text += f"- Model: {row['checkpoint_model']}\n\n"
    
    return best_text, worst_text

In [9]:
def search_ratings(min_aesthetic=0, model_filter="", sampler_filter=""):
    """Search and filter ratings"""
    df = db.get_all_ratings()
    
    if df.empty:
        return pd.DataFrame()
    
    # Apply filters
    filtered = df.copy()
    
    if min_aesthetic > 0:
        filtered = filtered[filtered['aesthetic_quality'] >= min_aesthetic]
    
    if model_filter:
        filtered = filtered[filtered['checkpoint_model'].str.contains(model_filter, case=False, na=False)]
    
    if sampler_filter:
        filtered = filtered[filtered['sampler'].str.contains(sampler_filter, case=False, na=False)]
    
    # Select relevant columns
    display_cols = [
        'image_path', 'aesthetic_quality', 'prompt_adherence',
        'technical_quality', 'style_consistency', 'checkpoint_model',
        'sampler', 'cfg_scale', 'seed'
    ]
    
    return filtered[display_cols]


In [10]:
def create_analysis_interface():
    with gr.Blocks(title="SD Rating Analysis", theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 📊 Image Rating Analysis Dashboard")
        
        with gr.Tabs():
            with gr.Tab("Summary"):
                gr.Markdown("## Overall Statistics")
                refresh_btn1 = gr.Button("🔄 Refresh Stats", size="sm")
                summary_output = gr.Markdown()
                
                refresh_btn1.click(
                    fn=get_summary_stats,
                    inputs=[],
                    outputs=[summary_output]
                )
                
                # Load initial stats
                demo.load(
                    fn=get_summary_stats,
                    inputs=[],
                    outputs=[summary_output]
                )
            
            with gr.Tab("Model Analysis"):
                gr.Markdown("## Performance by Checkpoint Model")
                refresh_btn2 = gr.Button("🔄 Refresh Model Stats", size="sm")
                model_output = gr.Dataframe()
                
                refresh_btn2.click(
                    fn=get_model_analysis,
                    inputs=[],
                    outputs=[model_output]
                )
                
                demo.load(
                    fn=get_model_analysis,
                    inputs=[],
                    outputs=[model_output]
                )
            
            with gr.Tab("LORA Analysis"):
                gr.Markdown("## Performance by LORA")
                refresh_btn3 = gr.Button("🔄 Refresh LORA Stats", size="sm")
                lora_output = gr.Dataframe()
                
                refresh_btn3.click(
                    fn=get_lora_analysis,
                    inputs=[],
                    outputs=[lora_output]
                )
                
                demo.load(
                    fn=get_lora_analysis,
                    inputs=[],
                    outputs=[lora_output]
                )
            
            with gr.Tab("Best/Worst"):
                gr.Markdown("## Top and Bottom Rated Images")
                refresh_btn4 = gr.Button("🔄 Refresh Rankings", size="sm")
                
                with gr.Row():
                    best_output = gr.Markdown()
                    worst_output = gr.Markdown()
                
                refresh_btn4.click(
                    fn=get_best_worst_images,
                    inputs=[],
                    outputs=[best_output, worst_output]
                )
                
                demo.load(
                    fn=get_best_worst_images,
                    inputs=[],
                    outputs=[best_output, worst_output]
                )
            
            with gr.Tab("Search"):
                gr.Markdown("## Search and Filter Ratings")
                
                with gr.Row():
                    min_aesthetic_input = gr.Slider(
                        minimum=0,
                        maximum=5,
                        value=0,
                        step=1,
                        label="Minimum Aesthetic Quality"
                    )
                    model_input = gr.Textbox(label="Model Filter (partial match)")
                    sampler_input = gr.Textbox(label="Sampler Filter (partial match)")
                
                search_btn = gr.Button("🔍 Search", variant="primary")
                search_output = gr.Dataframe()
                
                search_btn.click(
                    fn=search_ratings,
                    inputs=[min_aesthetic_input, model_input, sampler_input],
                    outputs=[search_output]
                )
    
    return demo

In [11]:
analysis_app = create_analysis_interface()
analysis_app.launch(share=False, inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
